In [1]:
# 1. 기존 잔재 삭제 및 MUSt3R 저장소 복제
%cd /content
!rm -rf must3r
!git clone --recursive https://github.com/naver/must3r.git
%cd must3r

# 2. 코랩 고유의 GPU PyTorch가 덮어씌워지는 것을 방지하기 위해 requirements 파일 수정
!sed -i '/torch/d' dust3r/requirements.txt
!sed -i '/torch/d' dust3r/requirements_optional.txt
!sed -i '/torch/d' requirements.txt

# 3. 필수 의존성 패키지 및 Gradio 라이브러리 설치
!pip install -r dust3r/requirements.txt
!pip install -r dust3r/requirements_optional.txt
!pip install -r requirements.txt
!pip install faiss-cpu cython gradio huggingface_hub viser

/content
Cloning into 'must3r'...
remote: Enumerating objects: 303, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 303 (delta 30), reused 31 (delta 24), pack-reused 246 (from 1)
Receiving objects: 100% (303/303), 601.32 KiB | 3.25 MiB/s, done.
Resolving deltas: 100% (170/170), done.
Submodule 'dust3r' (https://github.com/naver/dust3r) registered for path 'dust3r'
Cloning into '/content/must3r/dust3r'...
remote: Enumerating objects: 611, done.        
remote: Total 611 (delta 0), reused 0 (delta 0), pack-reused 611 (from 1)        
Receiving objects: 100% (611/611), 756.60 KiB | 3.08 MiB/s, done.
Resolving deltas: 100% (355/355), done.
Submodule path 'dust3r': checked out 'bb9f9f54826b4077ed9fd075308eb2fe2d4666f9'
Submodule 'croco' (https://github.com/naver/croco) registered for path 'dust3r/croco'
Cloning into '/content/must3r/dust3r/croco'...
remote: Enumerating objects: 198, done.        
remote: Counting objects: 10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 125.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 129.5 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.5 MB/s eta 0:00:00


In [1]:
# 1. ASMK 알고리즘 빌드 및 설치
%cd /content/must3r
!mkdir -p build
%cd build
!git clone https://github.com/jenicek/asmk.git
%cd asmk/cython/
!cythonize *.pyx
%cd ..
!pip install .

# 2. CuRoPE (위치 임베딩 가속 커널) 빌드
%cd /content/must3r/dust3r/croco/models/curope/
!pip install .

# 3. 메인 폴더로 복귀 및 GPU 상태 최종 검증
%cd /content/must3r

/content/must3r
/content/must3r/build
Cloning into 'asmk'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 138 (delta 78), reused 117 (delta 59), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 152.04 KiB | 1.14 MiB/s, done.
Resolving deltas: 100% (78/78), done.
/content/must3r/build/asmk/cython
Compiling /content/must3r/build/asmk/cython/hamming.pyx because it changed.
[1/1] Cythonizing /content/must3r/build/asmk/cython/hamming.pyx
/content/must3r/build/asmk
Processing /content/must3r/build/asmk
  Preparing metadata (setup.py) ... done
  Created wheel for asmk: filename=asmk-0.1-cp312-cp312-linux_x86_64.whl size=443435 sha256=3a56d7b7f7484cb375fce7dcdec15ea1e8617177c0fd6484bbfd223f77796a2e
  Stored in directory: /tmp/pip-ephem-wheel-cache-4tu_4s45/wheels/ea/68/64/6d589f7c177afbcab2ae6e5f1a986ca9d08f60f9d3ec7ac273
Successfully built asmk
/content/must3r/dust3r/croco/models

In [2]:
import torch
print('🔥 CUDA 이용 가능 여부:', torch.cuda.is_available())
print('🎮 사용 중인 GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')

🔥 CUDA 이용 가능 여부: True
🎮 사용 중인 GPU: Tesla T4


In [3]:
%cd /content/must3r
!mkdir -p checkpoints

# 512 해상도 메인 모델 및 이미지 검색용 가중치 다운로드
!wget -O checkpoints/MUSt3R_512.pth https://download.europe.naverlabs.com/ComputerVision/MUSt3R/MUSt3R_512.pth
!wget -O checkpoints/MUSt3R_512_retrieval_trainingfree.pth https://download.europe.naverlabs.com/ComputerVision/MUSt3R/MUSt3R_512_retrieval_trainingfree.pth

print("✅ 가중치 파일 준비 완료!")

/content/must3r
--2026-06-26 06:33:07--  https://download.europe.naverlabs.com/ComputerVision/MUSt3R/MUSt3R_512.pth
Resolving download.europe.naverlabs.com (download.europe.naverlabs.com)... 110.234.56.25
Connecting to download.europe.naverlabs.com (download.europe.naverlabs.com)|110.234.56.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1693926903 (1.6G)
Saving to: ‘checkpoints/MUSt3R_512.pth’

checkpoints/MUSt3R_ 100%[===================>]   1.58G  26.1MB/s    in 66s     

2026-06-26 06:34:14 (24.5 MB/s) - ‘checkpoints/MUSt3R_512.pth’ saved [1693926903/1693926903]

--2026-06-26 06:34:14--  https://download.europe.naverlabs.com/ComputerVision/MUSt3R/MUSt3R_512_retrieval_trainingfree.pth
Resolving download.europe.naverlabs.com (download.europe.naverlabs.com)... 110.234.56.25
Connecting to download.europe.naverlabs.com (download.europe.naverlabs.com)|110.234.56.25|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8399220 (8.0M)
Saving

In [5]:
import os

# 💡 코랩에 업로드한 본인의 동영상 파일 이름과 확장자로 수정해 주세요!
video_name = "KakaoTalk_20260626_150213971.mp4"
video_path = f"/content/{video_name}"

output_dir = "/content/must3r/video_frames"
fps = 3  # 초당 추출할 프레임 수 (동영상이 1분 이상으로 길다면 1~2로 낮추는 걸 추천합니다)

# 폴더 생성 및 기존 잔재 청소
!rm -rf "{output_dir}"
os.makedirs(output_dir, exist_ok=True)

# 리눅스 강력한 미디어 도구인 FFmpeg로 프레임 추출
if os.path.exists(video_path):
    print("🎬 동영상 변환 시작...")
    !ffmpeg -i "{video_path}" -vf "fps={fps}" "{output_dir}/frame_%04d.png" -loglevel quiet

    # 추출된 이미지 개수 확인
    num_frames = len([f for f in os.listdir(output_dir) if f.endswith('.png')])
    print(f"\n✅ 변환 완료! 총 {num_frames}개의 프레임이 추출되었습니다.")
    print(f"📍 저장된 경로: {output_dir}")

    if num_frames > 300:
        print("⚠️ [주의] 프레임이 300장을 넘으면 T4 GPU 메모리가 부족할 수 있습니다.")
        print("   영상을 더 짧게 자르거나, fps를 낮춰서 다시 실행하는 것을 권장합니다.")
else:
    print(f"❌ '{video_path}' 파일을 찾을 수 없습니다. 왼쪽 폴더 창에 파일이 잘 올라갔는지 확인해 주세요.")

🎬 동영상 변환 시작...

✅ 변환 완료! 총 40개의 프레임이 추출되었습니다.
📍 저장된 경로: /content/must3r/video_frames


In [9]:
target_path = "/content/must3r/must3r/demo/gradio.py"

with open(target_path, "r") as f:
    code = f.read()

# 꼬여버린 중복 인자 문장을 깔끔하게 하나로 통일
if "share=True, share=False" in code:
    code = code.replace("share=True, share=False", "share=True")
    with open(target_path, "w") as f:
        f.write(code)
    print("✅ 문법 오류(SyntaxError) 수정 완료! 이제 정상 작동합니다.")
else:
    print("💡 이미 수정되었거나 문장이 매칭되지 않습니다. 아래 데모를 실행해 보세요.")

✅ 문법 오류(SyntaxError) 수정 완료! 이제 정상 작동합니다.


In [11]:
%cd /content/must3r
!python demo.py \
    --weights checkpoints/MUSt3R_512.pth \
    --retrieval checkpoints/MUSt3R_512_retrieval_trainingfree.pth \
    --image_size 512 \
    --allow_local_files

/content/must3r
image_size 512 -> 512
Parsed pos_embed: RoPE100, base size = 512
image_size 512 -> 512
Parsed pos_embed: RoPE100, base size = 512
Outputing stuff in /tmp/tmph_0q04ludust3r_gradio_demo
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c1f5a892d0af79134f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
pointmaps_activation set to ActivationType.NORM_EXP
loading images
 - adding /content/must3r/video_frames/frame_0001.png with resolution 1080x1920 --> 288x512
 - adding /content/must3r/video_frames/frame_0002.png with resolution 1080x1920 --> 288x512
 - adding /content/must3r/video_frames/frame_0003.png with resolution 1080x1920 --> 288x512
 - adding /content/must3r/video_frames/frame_0004.png with resolution 1080x1920 --> 288x512
 - adding /content/must3r/video_frames/frame_